# Question Generation - Inference Notebook

**Use this notebook to:**
- Load the trained question generation model
- Test on video transcripts
- Generate questions in the format needed for Streamlit integration

## 1. Setup

In [ ]:
!pip install -q transformers accelerate peft bitsandbytes torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import json

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Load Model

In [ ]:
# Path to your trained model
MODEL_PATH = "/content/drive/MyDrive/question-generation-mistral"
BASE_MODEL = "mistralai/Mistral-7B-v0.1"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model with 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA weights...")
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()

print("\n✅ Model loaded successfully!")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Loading LoRA weights...

✅ Model loaded successfully!


## 4. Helper Functions

In [ ]:
def parse_generated_question(generated_text):
    """
    Parse model output into structured format for Streamlit

    Returns:
        dict: {"question": str, "choices": list, "correct": int}
    """
    try:
        if "### Question:" in generated_text:
            parts = generated_text.split("### Question:")[1]

            if "### Choices:" in parts:
                question_text = parts.split("### Choices:")[0].strip()
                choices_section = parts.split("### Choices:")[1]

                if "### Correct Answer:" in choices_section:
                    choices_text = choices_section.split("### Correct Answer:")[0].strip()
                    correct_answer = choices_section.split("### Correct Answer:")[1].strip()[0]

                    # Parse choices
                    choices = []
                    for line in choices_text.split('\n'):
                        if line.strip() and ')' in line:
                            choice = line.split(')', 1)[1].strip()
                            choices.append(choice)

                    # Convert answer letter to index
                    correct_idx = ord(correct_answer.upper()) - ord('A')

                    return {
                        "question": question_text,
                        "choices": choices,
                        "correct": correct_idx
                    }
    except Exception as e:
        print(f"Parse error: {e}")
        return None

def generate_single_question(transcript, temperature=0.7):
    """
    Generate one MCQ question from a transcript

    Args:
        transcript (str): Video transcript text
        temperature (float): Generation temperature (0.7 = balanced)

    Returns:
        dict: Parsed question or None
    """
    prompt = f"""### Passage:
{transcript.strip()}

### Task:
Generate a multiple-choice question based on the passage above.

### Question:"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=250,
            temperature=temperature,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return parse_generated_question(generated_text)

def generate_multiple_questions(transcript, num_questions=5, temperature=0.8):
    """
    Generate multiple MCQ questions from a transcript

    Args:
        transcript (str): Video transcript text
        num_questions (int): Number of questions to generate
        temperature (float): Higher = more variety (0.8 recommended)

    Returns:
        list: List of question dicts
    """
    questions = []

    for i in range(num_questions):
        print(f"Generating question {i+1}/{num_questions}...")

        prompt = f"""### Passage:
{transcript.strip()}

### Task:
Generate question #{i+1} - a multiple-choice question based on the passage above.

### Question:"""

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=250,
                temperature=temperature,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        parsed = parse_generated_question(generated_text)

        if parsed:
            questions.append(parsed)
        else:
            print(f"  ⚠️ Failed to parse question {i+1}")

    return questions

print("✅ Helper functions loaded")

✅ Helper functions loaded


## 5. Test with Sample Transcript

In [ ]:
# Sample transcript (replace with your own)
test_transcript = """
In this lecture, we'll discuss the concept of derivatives in calculus.
A derivative represents the rate of change of a function at a given point.
For example, if we have a function f(x) = x squared, the derivative is 2x.
This means that at any point x, the rate of change is twice the value of x.
Derivatives are fundamental in understanding how functions behave and are used
extensively in physics, engineering, and economics.
"""

print("Testing with sample transcript...\n")
question = generate_single_question(test_transcript)

print("="*80)
print("GENERATED QUESTION:")
print("="*80)
print(json.dumps(question, indent=2))
print("="*80)

Testing with sample transcript...

GENERATED QUESTION:
{
  "question": "Which of the following is TRUE?",
  "choices": [
    "Derivatives are only used in physics, engineering, and economics.",
    "The function f(x) = x squared is the derivative of the function 2x.",
    "The function 2x is the rate of change of the function f(x) = x squared.",
    "The rate of change of a function is twice the value of the function."
  ],
  "correct": 2
}


## 6. Generate Multiple Questions

In [ ]:
# Generate 5 questions
print("Generating 5 questions...\n")
questions = generate_multiple_questions(test_transcript, num_questions=5)

print("\n" + "="*80)
print(f"GENERATED {len(questions)} QUESTIONS:")
print("="*80)

for i, q in enumerate(questions, 1):
    print(f"\n📝 Question {i}:")
    print(f"   {q['question']}")
    print(f"\n   Choices:")
    for j, choice in enumerate(q['choices']):
        marker = "✅" if j == q['correct'] else "  "
        print(f"   {chr(65+j)}) {choice} {marker}")

print("\n" + "="*80)

Generating 5 questions...

Generating question 1/5...
Generating question 2/5...
Generating question 3/5...
Generating question 4/5...
Generating question 5/5...

GENERATED 5 QUESTIONS:

📝 Question 1:
   If the function is 2x, the rate of change at the point x=2 is   _  .

   Choices:
   A) 4   
   B) 8   
   C) 16   
   D) 2 ✅

📝 Question 2:
   Derivatives are used in   _  .

   Choices:
   A) all subjects   
   B) all sciences   
   C) most sciences   
   D) some sciences ✅

📝 Question 3:
   What is the rate of change of the function f(x)=x squared at any point x?

   Choices:
   A) 2x ✅
   B) x   
   C) 2x/x   
   D) 2x/2   

📝 Question 4:
   The passage is mainly about   _  .

   Choices:
   A) the definition of derivatives in calculus ✅
   B) the concept of physics in calculus   
   C) the application of derivatives in calculus   
   D) the importance of derivatives in calculus   

📝 Question 5:
   The main idea of this passage is   _   .

   Choices:
   A) the way of calculating 

## 7. Test with Your Own Transcript

In [ ]:
# Paste your own transcript here
my_transcript = """
Redis Cluster : sharding
Design : high perf. (scales to 1000 nodes) > best-effort write safety > availability
- Client-Server, TCP connection (Redis implements its own event library)
- Cluster nodes connected to each other (full mesh) through a TCP cluster bus
(gossip protocol/heartbeats to discover new nodes/detect failures).
- Redis assigns keys to nodes based hash slot (fixed hash function. Slot may
contain several k-v pairs).
- Each master (and its replicas) is responsible for a range of "hash slots".
- No proxys : clients can contact any server (included replicas) and the reply
indicates where to redirect query if needed
- Asynchronous replication (roughly at the same time as acknowledgement of a
write). Rule for merging conflicts is : last failover wins.
- Replicas are not attached to a single master. If master fails, replica replaces
master. When a master ends up without replica, a replica migrates from a
master with multiple replicas (the master that has most replicas) to an
orphaned master.
- After failure/split the part of cluster that has a minority of masters eventually
won’t accept writes, but may cause lost writes in the meantime.
"""

# Generate questions
my_questions = generate_multiple_questions(my_transcript, num_questions=5)

# Display
print("\n" + "="*80)
print("YOUR QUESTIONS:")
print("="*80)

for i, q in enumerate(my_questions, 1):
    print(f"\nQ{i}: {q['question']}")
    for j, choice in enumerate(q['choices']):
        marker = "✅" if j == q['correct'] else "  "
        print(f"  {chr(65+j)}) {choice} {marker}")

Generating question 1/5...
Generating question 2/5...
Generating question 3/5...
Generating question 4/5...
Generating question 5/5...

YOUR QUESTIONS:

Q1: What can we infer from the passage?
  A) Cluster nodes are connected to each other through TCP communication.   
  B) A cluster is designed to scale to 1000 nodes. ✅
  C) When the master fails, the client will be connected to the master's replicas.   
  D) Redis will use TCP event library to make a full mesh.   

Q2: When a master fails in Redis Cluster, a replica   _  .
  A) replaces it   
  B) contacts the other master   
  C) contacts the previous master   
  D) will cause lost writes in the meantime ✅

Q3: The following can be inferred from the passage EXCEPT that   _  .
  A) the cluster bus will help detect the failures of servers   
  B) the cluster bus is the Redis' own implementation   
  C) the cluster bus is a TCP network   
  D) the cluster bus is a data channel ✅

Q4: What’s the biggest problem for the Redis Cluster?
  

## 8. Save Questions to JSON

In [ ]:
# Save generated questions to file
output_file = "/content/drive/MyDrive/generated_questions.json"

with open(output_file, 'w') as f:
    json.dump(questions, f, indent=2)

print(f"✅ Questions saved to: {output_file}")
print(f"\nFormat: {json.dumps(questions[0], indent=2)}")

✅ Questions saved to: /content/drive/MyDrive/generated_questions.json

Format: {
  "question": "If the function is 2x, the rate of change at the point x=2 is   _  .",
  "choices": [
    "4",
    "8",
    "16",
    "2"
  ],
  "correct": 3
}


## 9. Integration Code for Streamlit

In [ ]:
integration_code = '''
# ============================================================================
# STREAMLIT INTEGRATION CODE
# Add this to your YouTube transcript app
# ============================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import json

# Load model once at startup
@st.cache_resource
def load_question_generation_model():
    """Load the fine-tuned question generation model"""

    MODEL_PATH = "path/to/question-generation-mistral"  # UPDATE THIS
    BASE_MODEL = "mistralai/Mistral-7B-v0.1"

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
    )

    model = PeftModel.from_pretrained(base_model, MODEL_PATH)
    model.eval()

    return model, tokenizer

def parse_question(generated_text):
    """Parse model output to structured format"""
    # [Same parsing function as above]
    pass

def generate_quiz(model, tokenizer, transcript, num_questions=5):
    """Generate quiz questions from transcript"""
    questions = []

    for i in range(num_questions):
        prompt = f"""### Passage:
{transcript}

### Task:
Generate question #{i+1} - a multiple-choice question.

### Question:"""

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=250,
                temperature=0.8,
                do_sample=True,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        parsed = parse_question(generated)

        if parsed:
            questions.append(parsed)

    return questions

# ============================================================================
# USAGE IN YOUR STREAMLIT APP:
# ============================================================================

# 1. Load model (runs once)
question_model, question_tokenizer = load_question_generation_model()

# 2. Get transcript (already working)
transcript = get_transcript(video_id)

# 3. Generate questions
questions = generate_quiz(question_model, question_tokenizer, transcript, num_questions=5)

# 4. Display quiz
st.subheader("📝 Quiz")
user_answers = []

for i, q in enumerate(questions):
    st.write(f"**Question {i+1}:** {q[\'question\']}")
    answer = st.radio(
        f"Select answer for Q{i+1}:",
        q[\'choices\'],
        key=f"q{i}"
    )
    user_answers.append(answer)

# 5. Evaluate (use your existing evaluation model)
if st.button("Submit Quiz"):
    score = evaluate_answers(questions, user_answers)
    st.write(f"Score: {score}/5")
'''

# Save integration code
with open('/content/drive/MyDrive/streamlit_integration.py', 'w') as f:
    f.write(integration_code)

print("✅ Integration code saved to: /content/drive/MyDrive/streamlit_integration.py")
print("\nCopy this code into your Streamlit app!")

✅ Integration code saved to: /content/drive/MyDrive/streamlit_integration.py

Copy this code into your Streamlit app!


---

## Summary

**This notebook provides:**
1. ✅ Load your trained question generation model
2. ✅ Generate questions from any transcript
3. ✅ Parse output into Streamlit-ready format
4. ✅ Integration code for your app

**Next steps:**
1. Download the model from Google Drive
2. Copy integration code to your Streamlit app
3. Connect transcript extraction → question generation → quiz display